In [1]:
import math
import random

import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("data/wifi_training.csv")

X = df[["Download", "Upload", "PacketLoss"]]
y = df[["Expected"]].rename(columns={"Expected": "O1"})

display(df.head())

,Download,Upload,PacketLoss,Expected
0,5,1,8.5,0
1,8,2,6.2,0
2,10,2,5.8,0
3,12,3,4.7,0
4,15,3,4.1,0


In [4]:
X_min = X.min()
X_max = X.max()

X_norm = (X - X_min) / (X_max - X_min).replace(0, 1)

X_norm.head()

,Download,Upload,PacketLoss
0,0.000000,0.000000,1.000000
1,0.020690,0.038462,0.729412
2,0.034483,0.038462,0.682353
3,0.048276,0.076923,0.552941
4,0.068966,0.076923,0.482353


In [6]:
class NeuralNetwork:
    def __init__(
        self,
        input_size=3,
        hidden_size=4,
        output_size=1,
        learning_rate=0.1,
    ):
        random.seed(42)
        self.learning_rate = learning_rate

        self.W1 = pd.DataFrame(
            [
                [random.gauss(0, 0.1) for _ in range(hidden_size)]
                for _ in range(input_size)
            ],
            index=["Download", "Upload", "PacketLoss"],
            columns=[f"H{i + 1}" for i in range(hidden_size)],
        )

        self.b1 = pd.Series(
            [0.0] * hidden_size,
            index=self.W1.columns,
        )

        self.W2 = pd.DataFrame(
            [
                [random.gauss(0, 0.1) for _ in range(output_size)]
                for _ in range(hidden_size)
            ],
            index=[f"H{i + 1}" for i in range(hidden_size)],
            columns=[f"O{i + 1}" for i in range(output_size)],
        )

        self.b2 = pd.Series(
            [0.0] * output_size,
            index=self.W2.columns,
        )

    def sigmoid(self, z):
        return z.map(
            lambda value: 1 / (1 + math.exp(-value))
        )

    def sigmoid_derivative(self, value):
        return value * (1 - value)

    def forward(self, X):
        self.z1 = X.dot(self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        self.z2 = self.a1.dot(self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)

        return self.a2

    def train(self, X, y, epochs=10000):
        for epoch in range(epochs):
            output = self.forward(X)

            error = y - output

            dz2 = error * self.sigmoid_derivative(output)

            dW2 = self.a1.T.dot(dz2)
            db2 = dz2.sum(axis=0)

            dz1 = pd.DataFrame(
                dz2.to_numpy() @ self.W2.to_numpy().T,
                index=dz2.index,
                columns=self.W2.index,
            ) * self.sigmoid_derivative(self.a1)

            dW1 = X.T.dot(dz1)
            db1 = dz1.sum(axis=0)

            self.W2 += self.learning_rate * dW2
            self.b2 += self.learning_rate * db2

            self.W1 += self.learning_rate * dW1
            self.b1 += self.learning_rate * db1

            if (epoch + 1) % 500 == 0:
                loss = (error ** 2).mean().mean()

                print(
                    f"Епоха {epoch + 1:5d}: "
                    f"помилка = {loss:.6f}"
                )

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        probability = self.predict_proba(X)

        return probability.map(
            lambda value: 1 if value >= 0.5 else 0
        )

In [7]:
network = NeuralNetwork()

network.train(X_norm, y, epochs=10000)

Епоха   500: помилка = 0.032195
Епоха  1000: помилка = 0.017846
Епоха  1500: помилка = 0.013503
Епоха  2000: помилка = 0.011207
Епоха  2500: помилка = 0.009664
Епоха  3000: помилка = 0.008516
Епоха  3500: помилка = 0.007610
Епоха  4000: помилка = 0.006868
Епоха  4500: помилка = 0.006244
Епоха  5000: помилка = 0.005710
Епоха  5500: помилка = 0.005247
Епоха  6000: помилка = 0.004840
Епоха  6500: помилка = 0.004481
Епоха  7000: помилка = 0.004160
Епоха  7500: помилка = 0.003873
Епоха  8000: помилка = 0.003614
Епоха  8500: помилка = 0.003379
Епоха  9000: помилка = 0.003166
Епоха  9500: помилка = 0.002972
Епоха 10000: помилка = 0.002795


In [8]:
probabilities = network.predict_proba(X_norm)["O1"]
predictions = network.predict(X_norm)["O1"]

training_results = X.copy()

training_results["Expected"] = df["Expected"].to_numpy()
training_results["Probability"] = probabilities.to_numpy()
training_results["Prediction"] = predictions.to_numpy()
training_results["Correct"] = (
    training_results["Expected"].to_numpy()
    == training_results["Prediction"].to_numpy()
)

training_results["Probability"] = training_results["Probability"].round(4)

display(training_results)

,Download,Upload,PacketLoss,Expected,Probability,Prediction,Correct
0,5,1,8.5,0,0.0000,0,True
1,8,2,6.2,0,0.0000,0,True
2,10,2,5.8,0,0.0000,0,True
3,12,3,4.7,0,0.0000,0,True
4,15,3,4.1,0,0.0000,0,True
5,18,4,3.8,0,0.0000,0,True
6,20,5,3.2,0,0.0000,0,True
7,22,4,2.9,0,0.0000,0,True
8,25,5,2.5,0,0.0000,0,True
9,28,6,2.1,0,0.0000,0,True


In [10]:
analysis_df = pd.read_csv("data/wifi_analysis.csv")

display(analysis_df.head())

,Download,Upload,PacketLoss
0,7,1,7.8
1,11,2,5.5
2,17,3,4.5
3,21,4,3.6
4,27,5,2.8


In [13]:
X_analysis = analysis_df[
    ["Download", "Upload", "PacketLoss"]
]

X_analysis_norm = (
    (X_analysis - X_min)
    / (X_max - X_min).replace(0, 1)
).fillna(0)

probabilities = network.predict_proba(X_analysis_norm)["O1"]
predictions = network.predict(X_analysis_norm)["O1"]

analysis_results = X_analysis.copy()

if "Expected" in analysis_df.columns:
    analysis_results["Expected"] = analysis_df["Expected"].to_numpy()
else:
    analysis_results["Expected"] = np.nan

analysis_results["Probability"] = probabilities.to_numpy()
analysis_results["Prediction"] = predictions.to_numpy()

if "Expected" in analysis_df.columns:
    analysis_results["Correct"] = (
        analysis_results["Expected"].to_numpy()
        == analysis_results["Prediction"].to_numpy()
    )
else:
    analysis_results["Correct"] = np.nan

analysis_results["Probability"] = analysis_results["Probability"].round(4)

display(analysis_results)

,Download,Upload,PacketLoss,Expected,Probability,Prediction,Correct
0,7,1,7.8,NaN,0.0000,0,NaN
1,11,2,5.5,NaN,0.0000,0,NaN
2,17,3,4.5,NaN,0.0000,0,NaN
3,21,4,3.6,NaN,0.0000,0,NaN
4,27,5,2.8,NaN,0.0000,0,NaN
5,33,6,2.0,NaN,0.0002,0,NaN
6,39,7,1.4,NaN,0.0270,0,NaN
7,42,6,2.5,NaN,0.0747,0,NaN
8,47,7,1.9,NaN,0.9251,1,NaN
9,52,8,1.6,NaN,0.9991,1,NaN
